# Train the leaf classifier on Colab

Backup way to train on a free T4 GPU. It clones the repo, downloads and builds
the dataset, trains, evaluates, and saves everything to your Google Drive.

Run the cells from top to bottom. Change the settings in the next cell first if you need to.

In [ ]:
BRANCH = "detection"          # or "main" once detection is merged
REPO_URL = "https://github.com/aishani0102/offline-crop-advisor.git"
ARCH = "large"                # "large" or "small"
EPOCHS = 15                   # total, the first FROZEN_EPOCHS train only the head
FROZEN_EPOCHS = 3
BATCH_SIZE = 128
NUM_WORKERS = 2               # Colab gives 2 CPU cores
DRIVE_DIR = "/content/drive/MyDrive/offline-crop-advisor"
DATA_ROOT = "/content/aisoc_data"
REPO_DIR = "/content/offline-crop-advisor"

## 1. Check there is a GPU

If this fails, go to Runtime > Change runtime type and pick T4 GPU.

In [ ]:
import subprocess, torch
print(subprocess.run(["nvidia-smi"], capture_output=True, text=True).stdout)
assert torch.cuda.is_available(), "no GPU, switch the runtime to T4"
print(torch.__version__, torch.cuda.get_device_name(0))

## 2. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
import os
os.makedirs(DRIVE_DIR + "/cache", exist_ok=True)
os.makedirs(DRIVE_DIR + "/runs", exist_ok=True)

## 3. Get the code

Colab already has PyTorch with CUDA, so we skip the torch and ExecuTorch lines
in the requirements file (export to .pte happens on the laptop).

In [ ]:
!rm -rf {REPO_DIR}
!git clone --depth 1 --branch {BRANCH} {REPO_URL} {REPO_DIR}
!git -C {REPO_DIR} log -1 --format="%h %s"
!grep -vE '^(torch|torchvision|torchao|executorch)==' {REPO_DIR}/detection/requirements.txt > /tmp/req_colab.txt
!pip install -q -r /tmp/req_colab.txt
!apt-get -qq install -y unar > /dev/null

## 4. Get the dataset

The first time, this downloads the six sources (about 1.5 GB, PlantSeg is the
slow one) and builds the dataset. The built dataset is then saved to Drive as
one zip, so after a reconnect it only unzips that. The build uses a fixed seed,
so it gives the same splits as the laptop. Compare the split fingerprint in
data_counts.md with the one in the repo to be sure.

In [ ]:
import os, shutil, subprocess

def sh(cmd):
    print(">", cmd)
    subprocess.run(cmd, shell=True, check=True)

cache_zip = f"{DRIVE_DIR}/cache/built.zip"
if os.path.exists(cache_zip):
    print("using cached dataset from Drive")
    os.makedirs(DATA_ROOT, exist_ok=True)
    sh(f"unzip -q -o {cache_zip} -d {DATA_ROOT}")
else:
    sh(f"python -I {REPO_DIR}/detection/data/download.py --root {DATA_ROOT}")
    sh(f"python -I {REPO_DIR}/detection/data/build_dataset.py --root {DATA_ROOT} --workers 2")
    shutil.copy(f"{REPO_DIR}/docs/data_counts.md", f"{DATA_ROOT}/built/data_counts.md")
    sh(f"cd {DATA_ROOT} && zip -q -r /content/built.zip built")
    shutil.copy("/content/built.zip", cache_zip)
    print("saved", cache_zip)
!grep -i fingerprint {DATA_ROOT}/built/data_counts.md
!grep -i fingerprint {REPO_DIR}/docs/data_counts.md

## 5. Train

The log shows seconds per step and per epoch. On a T4 one epoch should take a few minutes.

In [ ]:
RUN_DIR = f"/content/runs/mnv3_{ARCH}"
!python -I {REPO_DIR}/detection/train.py --root {DATA_ROOT} --out {RUN_DIR} --arch {ARCH} \
    --device cuda --batch-size {BATCH_SIZE} --num-workers {NUM_WORKERS} \
    --epochs {EPOCHS} --frozen-epochs {FROZEN_EPOCHS}

## 6. Evaluate on the lab test set and the field-only test set

In [ ]:
TAG = f"mnv3_{ARCH}_fp32_pytorch"
!python -I {REPO_DIR}/detection/evaluate.py --root {DATA_ROOT} --ckpt {RUN_DIR}/best.pt --tag {TAG}
!cat {REPO_DIR}/docs/results/{TAG}/metrics.md

## 7. Save the run to Drive

In [ ]:
import time, shutil, os
stamp = time.strftime("%Y%m%d_%H%M%S")
dest = f"{DRIVE_DIR}/runs/{stamp}_mnv3_{ARCH}"
os.makedirs(dest, exist_ok=True)
for f in ["best.pt", "temperature.json", "train_log.txt", "train_summary.json"]:
    shutil.copy(f"{RUN_DIR}/{f}", dest)
shutil.copytree(f"{REPO_DIR}/docs/results/{TAG}", f"{dest}/results_{TAG}")
shutil.copy(f"{DATA_ROOT}/built/data_counts.md", dest) if os.path.exists(f"{DATA_ROOT}/built/data_counts.md") else None
print("saved to", dest)
!ls -la {dest}

## 8. Back on the laptop

Export to .pte stays on the laptop. Download the run folder from Drive, then:

1. Put `best.pt` and `temperature.json` into `C:/Users/a_roy/aisoc_data/models/mnv3_large/`
   (one folder per run, the folder name becomes the .pte file name).
2. Make sure the laptop dataset is built with the same split fingerprint.
3. Run, from the repo folder:

```
python -I detection/evaluate.py --root C:/Users/a_roy/aisoc_data --ckpt C:/Users/a_roy/aisoc_data/models/mnv3_large/best.pt --tag mnv3_large_fp32_pytorch
python -I detection/export.py --root C:/Users/a_roy/aisoc_data --ckpt C:/Users/a_roy/aisoc_data/models/mnv3_large/best.pt
```